# Neural networks with PyTorch

Deep learning networks tend to be massive with dozens or hundreds of layers, that's where the term "deep" comes from. You can build one of these deep networks using only weight matrices, but in general it's very cumbersome and difficult to implement. PyTorch has a nice module `nn` that provides a nice way to efficiently build large neural networks.

In [1]:
%matplotlib inline

import numpy as np
import torch
import helper
import matplotlib.pyplot as plt


Now we're going to build a larger network that can solve a (formerly) difficult problem, identifying text in an image. Here we'll use the MNIST dataset which consists of greyscale handwritten digits. Each image is 28x28 pixels, you can see a sample below

<img src='assets/mnist.png'>

Our goal is to build a neural network that can take one of these images and predict the digit in the image.

First up, we need to get our dataset. This is provided through the `torchvision` package. The code below will download the MNIST dataset, then create training and test datasets for us. Don't worry too much about the details here, you'll learn more about this later.

**Obiettivo**

Vogliamo costruire una rete neurale per riconoscere cifre scritte a mano, cioè il dataset MNIST.MNIST contiene immagini in bianco e nero di cifre da 0 a 9.

Ogni immagine è 28 × 28 pixel,quindi ogni immagine contiene 28 * 28 = 784 valori

La rete deve prendere un’immagine e dire quale cifra rappresenta:

0, 1, 2, 3, 4, 5, 6, 7, 8, 9

Per questo l’output della rete avrà 10 neuroni, uno per ogni classe

In [ ]:
from torchvision import datasets, transforms
#torchvision è una libreria collegata a PyTorch utile per immagini e dataset.
# datasets contiene dataset già pronti, tra cui MNIST,
# transforms contiene trasformazioni da applicare alle immagini

# Define a transform to normalize the data
transform = transforms.Compose([transforms.ToTensor(),
                              transforms.Normalize((0.5,), (0.5,)),
                              ])
# Download and load the training data
trainset = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
#batch_size varia in base alla dimensione dell GPU che usiamo,è soggettivo
#batch_size = 64 vuol dire che prendo ogni volta 64 immagini
#transforms.ToTensor() converte l’immagine in un tensore PyTorch
#transforms.Normalize((0.5,), (0.5,)) normalizza i valori.
# In pratica sposta i pixel da circa 0 → 1 a circa -1 → 1
# Questo spesso aiuta la rete ad allenarsi meglio
#Prima ToTensor() porta i pixel circa tra 0 e 1:nero  → 0,bianco → 1
#Poi Normalize((0.5,), (0.5,)) li trasforma così:
#0   → (0 - 0.5) / 0.5 = -1
#0.5 → (0.5 - 0.5) / 0.5 = 0
#1   → (1 - 0.5) / 0.5 = 1


We have the training data loaded into `trainloader` and we make that an iterator with `iter(trainloader)`. Later, we'll use this to loop through the dataset for training, like

```python
for image, label in trainloader:
    ## do things with images and labels
```

You'll notice I created the `trainloader` with a batch size of 64, and `shuffle=True`. The batch size is the number of images we get in one iteration from the data loader and pass through our network, often called a *batch*. And `shuffle=True` tells it to shuffle the dataset every time we start going through the data loader again. But here I'm just grabbing the first batch so we can check out the data. We can see below that `images` is just a tensor with size `(64, 1, 28, 28)`. So, 64 images per batch, 1 color channel, and 28x28 images.

In [ ]:
dataiter = iter(trainloader)
#dataiter = iter(trainloader) trasforma trainloader in un iteratore.
images, labels = next(dataiter)
#images, labels = next(dataiter)prende un batch,images contiene le immagini,
# labels contiene le etichette corrette
print(type(images))
print(images.shape)
print(labels.shape)

This is what one of the images looks like. 

In [ ]:
plt.imshow(images[1].numpy().squeeze(), cmap='Greys_r');

First, let's try to build a simple network for this dataset using weight matrices and matrix multiplications. Then, we'll see how to do it using PyTorch's `nn` module which provides a much more convenient and powerful method for defining network architectures.

In the so-called *fully-connected* or *dense* networks, each unit in one layer is connected to each unit in the next layer. In fully-connected networks, the input to each layer must be a one-dimensional vector (which can be stacked into a 2D tensor as a batch of multiple examples). However, our images are 28x28 2D tensors, so we need to convert them into 1D vectors. Thinking about sizes, we need to convert the batch of images with shape `(64, 1, 28, 28)` to a have a shape of `(64, 784)`, 784 is 28 times 28. This is typically called *flattening*, we flattened the 2D images into 1D vectors.

Here we need 10 output units, one for each digit. We want our network to predict the digit shown in an image, so what we'll do is calculate probabilities that the image is of any one digit or class. This ends up being a discrete probability distribution over the classes (digits) that tells us the most likely class for the image. That means we need 10 output units for the 10 classes (digits). We'll see how to convert the network output into a probability distribution next.

> **Example:** Flatten the batch of images `images`. Then build a multi-layer network with 784 input units, 256 hidden units, and 10 output units using random tensors for the weights and biases. For now, use a sigmoid activation for the hidden layer. Leave the output layer without an activation, we'll add one that gives us a probability distribution next.

La rete sarà **fully connected**, cioè ogni neurone di un layer è collegato a ogni neurone del layer successivo.

Però le immagini sono 2D e sono 28 × 28

Una rete fully connected vuole vettori 1D.

Quindi dobbiamo fare il **flattening**.

**Flattening**

La forma originale del batch è:(64, 1, 28, 28)

cioè:
64 immagini

1 canale

28 righe

28 colonne

Dopo il flattening diventa:(64, 784)

Perché 28 * 28 = 784

Quindi ogni immagine diventa un vettore di 784 valori.

La rete richiesta è 784 input → 256 hidden → 10 output

Cioè:

784 input perché ogni immagine ha 784 pixel

256 neuroni nello strato nascosto

10 output perché ci sono 10 cifre possibili




In [ ]:
def activation(x):
    return 1/(1+torch.exp(-x)) #sigmoide

# Flatten the input images (64, 1, 28, 28) -> (64, 784)
inputs = images.view(images.shape[0], -1)
#Questa riga fa il flattening, images.shape[0] vale 64, perché abbiamo 64 immagini.
#Il -1 significa: PyTorch, calcola tu automaticamente la seconda dimensione

#inputs.shape = (64, 784)
#w1.shape     = (784, 256)
#(inputs · w1).shape = (64, 256)
#Risultato: per ogni immagine ottieni 256 valori, cioè i 256 neuroni hidden.

#b1 ha forma:(256) cioè un bias per ogni neurone hidden

# Create parameters
w1 = torch.randn(784, 256)
b1 = torch.randn(256)

w2 = torch.randn(256, 10)
b2 = torch.randn(10)

h = activation(torch.mm(inputs, w1) + b1)#h.shape = (64, 256)
#il numero di neuroni hidden è scelto da noi, quindi è un iperparametro
#z = torch.mm(inputs, w1) + b1
#h = activation(z)
#la z si chiama funzione di preattivazione,valore lineare prima dell'attivazione

out = torch.mm(h, w2) + b2#output.shape = (64, 10),
#64 righe  → 64 immagini,10 colonne → 10 classi/cifre
#in out ci sono punteggi grezzi, chiamati anche logits, dopo aver usato
#la funzione di attivazione softmax su out, i logits diventeranno probabilità
#prob = torch.softmax(out, dim=1)  # prob.shape = (64, 10)

#La funzione di attivazione serve a introdurre non linearità.

#Senza funzione di attivazione, anche se metti tanti layer, 
# la rete rimane praticamente una sola trasformazione lineare.
#La activation, invece, permette alla rete di imparare relazioni più complesse.
'''
out contiene 10 valori grezzi, uno per ogni cifra:
out[0] → punteggio per la cifra 0
out[1] → punteggio per la cifra 1
...
out[9] → punteggio per la cifra 9

Questi valori si chiamano spesso logits, cioè punteggi non ancora 
trasformati in probabilità.

Non si usa la sigmoide su out perché qui non abbiamo un problema binario,0 oppure 1

Abbiamo un problema multiclasse:0, 1, 2, 3, 4, 5, 6, 7, 8, 9

Quindi dopo out useremo la softmax, non la sigmoide.

La softmax trasforma i 10 punteggi in probabilità che sommano a 1.

Esempio:

out = [2.1, 0.5, -1.2, 3.4, ...]

dopo softmax diventa qualcosa tipo:

[0.10, 0.02, 0.01, 0.70, ...]

La classe con probabilità più alta è la predizione.
#ogni riga è una immagine delle 64 iniziali, ed ogni colonna mi dice con 
#quale probabilità l'immagine corrisponda ad un numero
#output[0, 0] = probabilità che la prima immagine sia uno 0
#output[0, 1] = probabilità che la prima immagine sia un 1
#output[0, 2] = probabilità che la prima immagine sia un 2
#...
#output[0, 9] = probabilità che la prima immagine sia un 9

inputs = matrice degli input, ogni riga è un’immagine.

inputs =
[
 immagine 1 con 784 pixel
 immagine 2 con 784 pixel
 immagine 3 con 784 pixel
 ...
 immagine 64 con 784 pixel
]

Quindi la prima riga di inputs viene moltiplicata con la prima colonna di w1,
il tutto sommato al primo elemento di b, 
su questo applico la sigmoide ed ottengo il primo neurone del hidden layer

Ogni riga di h corrisponde a una immagine del batch.

h[0]  → risultati hidden della prima immagine
h[1]  → risultati hidden della seconda immagine
h[2]  → risultati hidden della terza immagine
...
h[63] → risultati hidden della sessantaquattresima immagine

Quindi h[0] contiene 256 valori, 
cioè i 256 neuroni hidden calcolati per la prima immagine.

Ogni colonna di h corrisponde a un neurone hidden.

h[:, 0]   → primo neurone hidden per tutte le immagini
h[:, 1]   → secondo neurone hidden per tutte le immagini
h[:, 2]   → terzo neurone hidden per tutte le immagini
...
h[:, 255] → neurone hidden numero 256 per tutte le immagini

Quindi h[:, 0] contiene 64 valori, 
cioè il valore del primo neurone hidden calcolato su tutte le 64 immagini.

h[i, j] significa valore del neurone hidden j per l'immagine i

'''

Now we have 10 outputs for our network. We want to pass in an image to our network and get out a probability distribution over the classes that tells us the likely class(es) the image belongs to. Something that looks like this:

<img src='assets/image_distribution.png' width=500px>

Here we see that the probability for each class is roughly the same. This is representing an untrained network, it hasn't seen any data yet so it just returns a uniform distribution with equal probabilities for each class.

To calculate this probability distribution, we often use the [**softmax** function](https://en.wikipedia.org/wiki/Softmax_function). Mathematically this looks like

$$
\Large \sigma(x_i) = \cfrac{e^{x_i}}{\sum_k^K{e^{x_k}}}
$$

What this does is squish each input $x_i$ between 0 and 1 and normalizes the values to give you a proper probability distribution where the probabilites sum up to one.

## Building networks with PyTorch

PyTorch provides a module `nn` that makes building networks much simpler. Here I'll show you how to build the same one as above with 784 inputs, 256 hidden units, 10 output units and a softmax output.

**feedforward** = descrive la direzione del flusso dei dati
**fully connected** = descrive il tipo di collegamento tra due layer
La rete del notebook è entrambe le cose, è feedforward perché va da input a output senza tornare indietro,è fully connected perché usa layer nn.Linear

Per esempio, anche una **rete convoluzionale** può essere feedforward, ma non è completamente fully connected in tutti i suoi layer, perché usa convoluzioni invece di collegare tutto con tutto.

In [ ]:
from torch import nn

In [ ]:
class Network(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Inputs to hidden layer linear transformation
        self.hidden = nn.Linear(784, 256)
        # Output layer, 10 units - one for each digit
        self.output = nn.Linear(256, 10)
        
        # Define sigmoid activation and softmax output 
        self.sigmoid = nn.Sigmoid()
        self.softmax = nn.Softmax(dim=1)
        
    def forward(self, x):
        # Pass the input tensor through each of our operations
        # Se x arriva come (64, 1, 28, 28), lo trasformo in (64, 784) con .view
        x = x.view(x.shape[0], -1)
        x = self.hidden(x)
        x = self.sigmoid(x)
        x = self.output(x)
        x = self.softmax(x)
        
        return x
    
#per funzionare x.shape deve essere (batch_size, 784), nel nostro caso
#batch_size = 64, deciso precedentemente nel DataLoader, il tutto non 
#non funziona se x.shape = (64, 1, 28, 28),
# perché nn.Linear(784, 256) vuole 784 valori per immagine, non una matrice 28×28.
#Per sicurezza si può mettere una .view che trasforma la shape
#view in PyTorch serve a cambiare la forma di un tensore, 
# senza cambiare i valori contenuti dentro.
#x.view(x.shape[0], -1)
#mantieni il numero di immagini: images.shape[0] = 64
#-1 significa appiattisci tutto il resto
#esempio: passo x.shape=(64,1,28,28) la view la fa diventare x.shape=(64,784)


Let's go through this bit by bit.

```python
class Network(nn.Module):
```

Here we're inheriting from `nn.Module`. Combined with `super().__init__()` this creates a class that tracks the architecture and provides a lot of useful methods and attributes. It is mandatory to inherit from `nn.Module` when you're creating a class for your network. The name of the class itself can be anything.

```python
self.hidden = nn.Linear(784, 256)
```

This line creates a module for a linear transformation, $x\mathbf{W} + b$, with 784 inputs and 256 outputs and assigns it to `self.hidden`. The module automatically creates the weight and bias tensors which we'll use in the `forward` method. You can access the weight and bias tensors once the network (`net`) is created with `net.hidden.weight` and `net.hidden.bias`.

```python
self.output = nn.Linear(256, 10)
```

Similarly, this creates another linear transformation with 256 inputs and 10 outputs.

```python
self.sigmoid = nn.Sigmoid()
self.softmax = nn.Softmax(dim=1)
```

Here I defined operations for the sigmoid activation and softmax output. Setting `dim=1` in `nn.Softmax(dim=1)` calculates softmax across the columns.

```python
def forward(self, x):
```

PyTorch networks created with `nn.Module` must have a `forward` method defined. It takes in a tensor `x` and passes it through the operations you defined in the `__init__` method.

```python
x = self.hidden(x)
x = self.sigmoid(x)
x = self.output(x)
x = self.softmax(x)
```

Here the input tensor `x` is passed through each operation a reassigned to `x`. We can see that the input tensor goes through the hidden layer, then a sigmoid function, then the output layer, and finally the softmax function. It doesn't matter what you name the variables here, as long as the inputs and outputs of the operations match the network architecture you want to build. The order in which you define things in the `__init__` method doesn't matter, but you'll need to sequence the operations correctly in the `forward` method.

Now we can create a `Network` object.

In [ ]:
# Create the network and look at it's text representation
model = Network()
model

You can define the network somewhat more concisely and clearly using the `torch.nn.functional` module. This is the most common way you'll see networks defined as many operations are simple element-wise functions. We normally import this module as `F`, `import torch.nn.functional as F`.

In [ ]:
import torch.nn.functional as F
#altro modo di scrivere la rete precedentemente vista
class Network(nn.Module):
    def __init__(self):
        super().__init__()
        # Inputs to hidden layer linear transformation
        self.hidden = nn.Linear(784, 256)
        # Output layer, 10 units - one for each digit
        self.output = nn.Linear(256, 10)
        
    def forward(self, x):
        x=x.view(x.shape[0],-1)#sempre per sicurezza
        # Hidden layer with sigmoid activation
        x = F.sigmoid(self.hidden(x))
        # Output layer with softmax activation
        x = F.softmax(self.output(x), dim=1)
        
        return x
    
#nel codice non si vedono matrici dei pesi e bias, 
# perché li sta creando PyTorch dentro nn.Linear.
#Quando scrivi: self.hidden = nn.Linear(784, 256)
#PyTorch crea automaticamente:
#self.hidden.weight
#self.hidden.bias

#cioè i pesi e i bias del primo layer.

#Quando scrivi:self.output = nn.Linear(256, 10)
#PyTorch crea automaticamente:
#self.output.weight
#self.output.bias

In [ ]:
test = Network()
test

### Activation functions

So far we've only been looking at the softmax activation, but in general any function can be used as an activation function. The only requirement is that for a network to approximate a non-linear function, the activation functions must be non-linear. Here are a few more examples of common activation functions: Tanh (hyperbolic tangent), and ReLU (rectified linear unit).

<img src="assets/activation.png" width=700px>

In practice, the ReLU function is used almost exclusively as the activation function for hidden layers.

### Your Turn to Build a Network

<img src="assets/mlp_mnist.png" width=600px>

> **Example:** Create a network with 784 input units, a hidden layer with 128 units and a ReLU activation, then a hidden layer with 64 units and a ReLU activation, and finally an output layer with a softmax activation as shown above. You can use a ReLU activation with the `nn.ReLU` module or `F.relu` function.

It's good practice to name your layers by their type of network, for instance 'fc' to represent a fully-connected layer. As you code your solution, use `fc1`, `fc2`, and `fc3` as your layer names.

In [ ]:
class Network(nn.Module):
    def __init__(self):
        super().__init__()
        # Defining the layers, 128, 64, 10 units each
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        # Output layer, 10 units - one for each digit
        self.fc3 = nn.Linear(64, 10)
        
    def forward(self, x):
        ''' Forward pass through the network, returns the output logits '''
        
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        x = F.softmax(x, dim=1)
        
        return x

model = Network()
model

### Initializing weights and biases

The weights and such are automatically initialized for you, but it's possible to customize how they are initialized. The weights and biases are tensors attached to the layer you defined, you can get them with `model.fc1.weight` for instance.

In [ ]:
print(model.fc1.weight)
print(model.fc1.bias)

For custom initialization, we want to modify these tensors in place. These are actually autograd *Variables*, so we need to get back the actual tensors with `model.fc1.weight.data`. Once we have the tensors, we can fill them with zeros (for biases) or random normal values.

In [ ]:
# Set biases to all zeros
model.fc1.bias.data.fill_(0)

In [ ]:
# sample from random normal with standard dev = 0.01
model.fc1.weight.data.normal_(std=0.01)

### Forward pass

Now that we have a network, let's see what happens when we pass in an image.

In [ ]:
# Grab some data 
dataiter = iter(trainloader)
images, labels = next(dataiter)

# Resize images into a 1D vector, new shape is (batch size, color channels, image pixels) 
images.resize_(64, 1, 784)
# or images.resize_(images.shape[0], 1, 784) to automatically get batch size

# Forward pass through the network
img_idx = 0
ps = model.forward(images[img_idx,:])

img = images[img_idx]
helper.view_classify(img.view(1, 28, 28), ps)
#Questo codice prende una sola immagine dal batch, 
# la passa nella rete, ottiene le probabilità ps, 
# poi visualizza immagine + probabilità.
#img_idx = 0,è l'indice della prima riga, quindi la prima immagine
#images[img_idx, :] prende l’immagine numero img_idx
#ps = model.forward(images[img_idx,:]) qui passo quella immagine dentro la rete,
#e calcolo le probabilità per le 10 cifre:
'''
ps[0] → probabilità cifra 0
ps[1] → probabilità cifra 1
ps[2] → probabilità cifra 2
...
ps[9] → probabilità cifra 9
'''

As you can see above, our network has basically no idea what this digit is. It's because we haven't trained it yet, all the weights are random!

### Using `nn.Sequential`

PyTorch provides a convenient way to build networks like this where a tensor is passed sequentially through operations, `nn.Sequential` ([documentation](https://pytorch.org/docs/master/nn.html#torch.nn.Sequential)). Using this to build the equivalent network:

In [ ]:
# Hyperparameters for our network
input_size = 784
hidden_sizes = [128, 64]
output_size = 10

# Build a feed-forward network
model = nn.Sequential(nn.Linear(input_size, hidden_sizes[0]),
                      nn.ReLU(),
                      nn.Linear(hidden_sizes[0], hidden_sizes[1]),
                      nn.ReLU(),
                      nn.Linear(hidden_sizes[1], output_size),
                      nn.Softmax(dim=1))
print(model)

# Forward pass through the network and display output
images, labels = next(iter(trainloader))
images.resize_(images.shape[0], 1, 784)
ps = model.forward(images[0,:])
helper.view_classify(images[0].view(1, 28, 28), ps)

The operations are availble by passing in the appropriate index. For example, if you want to get first Linear operation and look at the weights, you'd use `model[0]`.

In [ ]:
print(model[0])
model[0].weight

You can also pass in an `OrderedDict` to name the individual layers and operations, instead of using incremental integers. Note that dictionary keys must be unique, so _each operation must have a different name_.

In [ ]:
from collections import OrderedDict
model = nn.Sequential(OrderedDict([
                      ('fc1', nn.Linear(input_size, hidden_sizes[0])),
                      ('relu1', nn.ReLU()),
                      ('fc2', nn.Linear(hidden_sizes[0], hidden_sizes[1])),
                      ('relu2', nn.ReLU()),
                      ('output', nn.Linear(hidden_sizes[1], output_size)),
                      ('softmax', nn.Softmax(dim=1))]))
model

Now you can access layers either by integer or the name

In [ ]:
print(model[0])
print(model.fc1)